# SVM Assignment


In [1]:
lazy import pandas as pd
lazy import numpy as np
lazy import matplotlib.pyplot as plt
lazy import seaborn as sns
lazy import joblib

In [2]:
# 1. Load the ransomware dataset and display first rows, rows and columns
df = pd.read_csv('ransomware-dataset.csv')
print('Shape:', df.shape)
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
print(df.head())


Shape: (62485, 18)
Rows: 62485
Columns: 18
                                            FileName  \
0           0124e21d-018c-4ce0-92a3-b9e205a76bc0.dll   
1       05c8318f98a5d301d80000009c316005.vertdll.dll   
2           06054fba-5619-4a86-a861-ffb0464bef5d.dll   
3        075822ac99a5d301660400009c316005.adhapi.dll   
4  090607dd9ba5d301ca0900009c316005.SensorsNative...   

                            md5Hash  Machine  DebugSize  DebugRVA  \
0  79755c51e413ed3c6be4635fd729a6e1      332          0         0   
1  95e19f3657d34a432eada93221b0ea16    34404         84    121728   
2  85c32641d77a54e19ba8ea4ab305c791      332          0         0   
3  62e3b959d982ef534b66f819fe15f085    34404         84     19904   
4  ae38c5f7d313ad0ff3bfb8826476767f    34404         84     97728   

   MajorImageVersion  MajorOSVersion  ExportRVA  ExportSize  IatVRA  \
0                  0               4          0           0    8192   
1                 10              10     126576        4930    

In [3]:
# 2. Check for missing values and clean the data
print(df.isnull().sum().sum())
print(df.isnull().sum())

df = df.dropna(axis=1, how='all')
for col in df.columns:
    if df[col].isnull().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])
print('Missing values after cleaning:', df.isnull().sum().sum())


0
FileName              0
md5Hash               0
Machine               0
DebugSize             0
DebugRVA              0
MajorImageVersion     0
MajorOSVersion        0
ExportRVA             0
ExportSize            0
IatVRA                0
MajorLinkerVersion    0
MinorLinkerVersion    0
NumberOfSections      0
SizeOfStackReserve    0
DllCharacteristics    0
ResourceSize          0
BitcoinAddresses      0
Benign                0
dtype: int64
Missing values after cleaning: 0


In [4]:
# 3. Display dtypes and convert non-numeric columns to numeric
print(df.dtypes)

for col in df.columns:
    if not pd.api.types.is_numeric_dtype(df[col]):
        print(f'Encoding column: {col}')
        df[col] = df[col].astype(str)
        df[col] = pd.Categorical(df[col]).codes
print(df.dtypes)


FileName                str
md5Hash                 str
Machine               int64
DebugSize             int64
DebugRVA              int64
MajorImageVersion     int64
MajorOSVersion        int64
ExportRVA             int64
ExportSize            int64
IatVRA                int64
MajorLinkerVersion    int64
MinorLinkerVersion    int64
NumberOfSections      int64
SizeOfStackReserve    int64
DllCharacteristics    int64
ResourceSize          int64
BitcoinAddresses      int64
Benign                int64
dtype: object
Encoding column: FileName
Encoding column: md5Hash
FileName              int32
md5Hash               int32
Machine               int64
DebugSize             int64
DebugRVA              int64
MajorImageVersion     int64
MajorOSVersion        int64
ExportRVA             int64
ExportSize            int64
IatVRA                int64
MajorLinkerVersion    int64
MinorLinkerVersion    int64
NumberOfSections      int64
SizeOfStackReserve    int64
DllCharacteristics    int64
ResourceSiz

In [5]:
# 4. Identify the target variable and split features/target
target_col = 'Label' if 'Label' in df.columns else df.columns[-1]
print('Target column:', target_col)
X = df.drop(columns=[target_col])
y = df[target_col]
print('X shape:', X.shape)
print('y shape:', y.shape)


Target column: Benign
X shape: (62485, 17)
y shape: (62485,)


In [6]:
from sklearn.model_selection import train_test_split

# 5. Train/test split in 80:20 ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)


(49988, 17) (12497, 17)
(49988,) (12497,)


In [7]:
from sklearn.preprocessing import StandardScaler

# 6. Standard scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Scaled train mean approx 0:', X_train_scaled.mean().round(4))
print('Scaled train std approx 1:', X_train_scaled.std().round(4))


Scaled train mean approx 0: 0.0
Scaled train std approx 1: 1.0


In [8]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 7. Train SVM with RBF kernel and evaluate
svm_rbf = SVC(kernel='rbf', random_state=42)
svm_rbf.fit(X_train_scaled, y_train)
y_pred_rbf = svm_rbf.predict(X_test_scaled)
acc_rbf = accuracy_score(y_test, y_pred_rbf)
print('RBF accuracy:', acc_rbf)


RBF accuracy: 0.9974393854525087


In [9]:
from sklearn.metrics import confusion_matrix, classification_report

# 8. Confusion matrix and classification report
cm = confusion_matrix(y_test, y_pred_rbf)
print('Confusion Matrix:')
print(cm)
print(classification_report(y_test, y_pred_rbf))


Confusion Matrix:
[[7066    7]
 [  25 5399]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7073
           1       1.00      1.00      1.00      5424

    accuracy                           1.00     12497
   macro avg       1.00      1.00      1.00     12497
weighted avg       1.00      1.00      1.00     12497



In [10]:
from sklearn.model_selection import RandomizedSearchCV

# 9. Hyperparameter tuning using Random Search
param_dist = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 0.01, 0.1],
    'kernel': ['rbf', 'linear',]
}

random_search = RandomizedSearchCV(
    estimator=SVC(),
    param_distributions=param_dist,
    n_iter=6,
    cv=3,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=2
)

random_search.fit(X_train_scaled, y_train)
print('Best params:', random_search.best_params_)
print('Best cross-val accuracy:', random_search.best_score_)


Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best params: {'kernel': 'rbf', 'gamma': 'scale', 'C': 0.1}
Best cross-val accuracy: 0.9881971188865007


In [11]:
from sklearn.metrics import accuracy_score

# 10. Train SVM with linear kernel and compare performance
svm_linear = SVC(kernel='linear', random_state=42)
svm_linear.fit(X_train_scaled, y_train)
y_pred_linear = svm_linear.predict(X_test_scaled)
acc_linear = accuracy_score(y_test, y_pred_linear)
print('Linear accuracy:', acc_linear)
print('RBF accuracy:', acc_rbf)
print('Difference:', acc_rbf - acc_linear)


Linear accuracy: 0.9046171081059454
RBF accuracy: 0.9974393854525087
Difference: 0.09282227734656323


In [12]:
from sklearn.metrics import accuracy_score

# 11. Compare performance with and without feature scaling
svm_no_scale = SVC(kernel='rbf', random_state=42)
svm_no_scale.fit(X_train, y_train)
y_pred_no_scale = svm_no_scale.predict(X_test)
acc_no_scale = accuracy_score(y_test, y_pred_no_scale)
print('Without scaling accuracy:', acc_no_scale)
print('With scaling accuracy:', acc_rbf)
print('Observed difference:', acc_rbf - acc_no_scale)


Without scaling accuracy: 0.7841081859646315
With scaling accuracy: 0.9974393854525087
Observed difference: 0.21333119948787715


In [13]:
from sklearn.metrics import accuracy_score

# 12. Save the trained model to disk and reload it
joblib.dump(svm_rbf, 'svm_rbf_model.joblib')
loaded_model = joblib.load('svm_rbf_model.joblib')
reloaded_pred = loaded_model.predict(X_test_scaled)
print('Reloaded model accuracy:', accuracy_score(y_test, reloaded_pred))


Reloaded model accuracy: 0.9974393854525087


In [14]:
from sklearn.model_selection import cross_val_score

# 13. Cross-validation for SVM
cv_scores = cross_val_score(SVC(kernel='rbf', random_state=42), X_train_scaled, y_train, cv=5, scoring='accuracy')
print('Cross-validation scores:', cv_scores)
print('Mean accuracy:', cv_scores.mean())
print('Std deviation:', cv_scores.std())


Cross-validation scores: [0.9974995  0.9969994  0.99679936 0.99739922 0.99729919]
Mean accuracy: 0.9971993338349575
Std deviation: 0.0002607596650416193


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 14. Pipeline with scaling + SVM
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', random_state=42))
])
pipeline.fit(X_train, y_train)
pipe_pred = pipeline.predict(X_test)
print('Pipeline accuracy:', accuracy_score(y_test, pipe_pred))


Pipeline accuracy: 0.9974393854525087


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 15. Sensitivity analysis for different train/test splits
split_ratios = [(0.6, 0.4), (0.7, 0.3), (0.9, 0.1)]
for train_frac, test_frac in split_ratios:
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
        X, y, test_size=test_frac, train_size=train_frac, random_state=42, stratify=y
    )
    scaler_s = StandardScaler()
    X_train_scaled_s = scaler_s.fit_transform(X_train_s)
    X_test_scaled_s = scaler_s.transform(X_test_s)
    svm_s = SVC(kernel='rbf', random_state=42)
    svm_s.fit(X_train_scaled_s, y_train_s)
    pred_s = svm_s.predict(X_test_scaled_s)
    acc_s = accuracy_score(y_test_s, pred_s)
    print(f'Split {train_frac}:{test_frac} -> Accuracy: {acc_s:.4f}')


Split 0.6:0.4 -> Accuracy: 0.9972
Split 0.7:0.3 -> Accuracy: 0.9979
Split 0.9:0.1 -> Accuracy: 0.9973
